# 05 Backtesting & EVT Analysis
Evaluating the VaR model and performing tail validation.

In [ ]:
import pandas as pd
from src.backtesting_tests import kupiec_pof_test, christoffersen_ind_test, adjust_var_rl
from src.evt_tail_analysis import fit_gpd_to_tails, plot_evt_tail
from src.visualization import plot_var_results
from src.ddqn_agent import DDQNAgent

modeling = pd.read_csv('../data/processed/modeling_results.csv', index_col=0, parse_dates=True)
X_test = pd.read_csv('../data/processed/selected_features.csv', index_col=0, parse_dates=True)
X_test = X_test.join(modeling[['vol_GARCH', 'vol_GJR']], how='inner')
X_test = X_test[X_test.index > '2022-12-31']

agent = DDQNAgent(X_test.shape[1], 2, 1.0)
agent.load('../models/saved_models/ddqn_agent.pth')
agent.epsilon = 0

test_returns = pd.read_csv('../data/processed/log_returns.csv', index_col=0, parse_dates=True)['^STOXX50E'].loc[X_test.index]
var_garch = modeling['VaR_GARCH_5%'].loc[X_test.index]

predictions = [agent.act(X_test.iloc[i].values) for i in range(len(X_test))]
var_rl = pd.Series([adjust_var_rl(v, p) for v, p in zip(var_garch, predictions)], index=X_test.index)

print("Kupiec Test (GARCH):", kupiec_pof_test(test_returns, var_garch))
print("Kupiec Test (RL):", kupiec_pof_test(test_returns, var_rl))

evt_res = fit_gpd_to_tails(test_returns)
print("EVT KS p-value:", evt_res['ks_pvalue'])
plot_evt_tail(evt_res, '../results/plots/evt_tail_fit.png')

plot_var_results(test_returns, var_garch, var_rl, 'VaR Comparison (2023-2025)', '../results/plots/final_var_comparison.png')